In [3]:
import os
import pandas as pd

**Step 1**
Filtering .narrowpeak files: 
1. Filter rows where Fold Change (FC) (column 7) is greater than 5.
2. Filter rows where the -log10(p-value) (column 8) is greater than 10.
3. Filter rows where the Fold Change (FC) is greater than 5% of the maximum Fold Change from the filtered results.

In [4]:
# Define the directory where the .narrowPeak files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/"  # Change this to the actual directory

# Get the list of files in the directory that end with .narrowPeak
narrowpeak_files = [f for f in os.listdir(directory) if f.endswith(".narrowPeak")]

# Initialize a list to store the results
summary_data = []

# Iterate over each .narrowPeak file
for filename in narrowpeak_files:
    # Read the file with unnamed columns
    file_path = os.path.join(directory, filename)
    
    # Check if the file is empty
    if os.stat(file_path).st_size == 0:
        print(f"The file {filename} is empty. It will be skipped.")
        continue

    try:
        file = pd.read_csv(file_path, sep="\t", header=None)
    except pd.errors.EmptyDataError:
        print(f"The file {filename} does not contain valid data. It will be skipped.")
        continue

    # Assign a list of names to the columns
    file.columns = ["Chromosome", "Start", "End", "Name", "Score", "Unused", "FC", "pValue", "qValue", "Peak"]

    # Count the number of rows before filtering
    initial_count = len(file)

    # Filter the rows where the value in the 'FC' column is greater than 1.5
    filtered_file = file[file["FC"] > 1.5]
    
    # Filter the 'pValue' column to be greater than 10
    filtered_file = filtered_file[filtered_file["pValue"] > 10]

    # Get the maximum value from the 'FC' column
    max_fc = filtered_file["FC"].max()

    # Calculate 5% of that maximum value
    threshold = max_fc * 0.05

    # Get only the rows that have 'FC' greater than 5% of the maximum value
    filtered_file = filtered_file[filtered_file["FC"] > threshold]

    # Count the number of rows after filtering
    filtered_count = len(filtered_file)

    # Add the results to the list
    summary_data.append([filename, initial_count, filtered_count])

    # Define the output file name
    output_filename = filename.replace(".narrowPeak", "_filtered.narrowPeak")
    output_path = os.path.join(directory, output_filename)

    # Save the filtered DataFrame to a new CSV file
    filtered_file.to_csv(output_path, sep="\t", index=False, header=False)

    print(f"Filtered file saved: {output_path}")

# Convert the results into a DataFrame
summary_df = pd.DataFrame(summary_data, columns=["Filename", "Initial Rows", "Filtered Rows"])

# Save the summary table to a CSV file
summary_output_path = os.path.join(directory, "filter_summary.csv")
summary_df.to_csv(summary_output_path, index=False)

print(f"Summary saved to: {summary_output_path}")


Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1762580_HSF1_filtered.narrowPeak
Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1789727_FK1_filtered.narrowPeak
Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1799646_Znf1_filtered.narrowPeak
Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1867537_MADS1_filtered.narrowPeak
Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1873322_HLH1_filtered.narrowPeak
Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1873386_HLH2_filtered.narrowPeak
Filtered file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1874682_HOX1_filtered.narrowPeak

**Step 2**
Filtering _filtered.narrowpeak files: 

1. Iterate Over Filtered Files: Identify all files in the specified directory that end with _filtered.narrowPeak.

2. Read Each Filtered File: For each file, read its contents into a DataFrame using pandas.

3. Calculate Length for Each Row: For each row in the DataFrame, calculate the difference between the End and Start columns:

     Length = End - Start
4. Calculate Half Length: Divide the calculated length by 5 and then divide the result by 2:

    Half Length = (Length / 5) / 2
5. Update Start and End Columns:

    New Start = Start + Half Length
    New End = End - Half Length
6. Save the Updated DataFrame: Create a new file with the name pattern _filtered_cut.narrowPeak and save the updated DataFrame.

7. Repeat for All Filtered Files: Repeat the above steps for all files that match the _filtered.narrowPeak naming convention.

In [6]:

# Define the directory where the filtered .narrowPeak files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/"  # Change this to the actual directory

# Get the list of files in the directory that end with filtered.narrowPeak
filtered_files = [f for f in os.listdir(directory) if f.endswith("filtered.narrowPeak")]

# Iterate over each filtered.narrowPeak file
for filename in filtered_files:
    file_path = os.path.join(directory, filename)
    
    # Check if the file is not empty
    if os.path.getsize(file_path) > 0:
        try:
            # Read the file with unnamed columns
            file = pd.read_csv(file_path, sep="\t", header=None)
            
            # Assign a list of names to the columns
            file.columns = ["Chromosome", "Start", "End", "Name", "Score", "Unused", "FC", "pValue", "qValue", "Peak"]
            
            # Calculate the new value for Start and End
            length = round((file["End"] - file["Start"]) / 5)
            length = length // 2  # Divide by 2 and ensure the result is an integer
            
            # Update the Start and End columns and convert to integers
            file["Start"] = (file["Start"] + length).astype(int)
            file["End"] = (file["End"] - length).astype(int)
            
            # Define the output file name
            output_filename = filename.replace("filtered.narrowPeak", "filtered_cut.narrowPeak")
            output_path = os.path.join(directory, output_filename)
            
            # Save the modified DataFrame to a new CSV file
            file.to_csv(output_path, sep="\t", index=False, header=False)
            
            print(f"Adjusted file saved: {output_path}")
        
        except pd.errors.EmptyDataError:
            print(f"Empty or unreadable file: {filename}, skipping...")
    else:
        print(f"Empty file: {filename}, skipping...")


Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1762580_HSF1_filtered_cut.narrowPeak
Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1789727_FK1_filtered_cut.narrowPeak
Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1799646_Znf1_filtered_cut.narrowPeak
Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1867537_MADS1_filtered_cut.narrowPeak
Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1873322_HLH1_filtered_cut.narrowPeak
Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1873386_HLH2_filtered_cut.narrowPeak
Adjusted file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Rhimi59_2_1874682_